In [16]:
import requests
import time
import random


def fetch_ign_elevations_node_by_node(nodes_gdf):
    nodes = nodes_gdf.to_crs(4326).copy()

    elevations = []

    url = "https://data.geopf.fr/altimetrie/1.0/calcul/alti/rest/elevation.json"

    for idx, row in nodes.iterrows():

        lon = row["x"]
        lat = row["y"]

        params = {
            "lon": str(lon),
            "lat": str(lat),
            "resource": "ign_rge_alti_wld",
            "delimiter": "|",
            "indent": "false",
            "measures": "false",
            "zonly": "false",
        }

        success = False

        for attempt in range(6):

            try:
                r = requests.get(url, params=params, timeout=30)

                if r.status_code == 429:
                    raise Exception("rate limit")

                r.raise_for_status()

                z = r.json()["elevations"][0]["z"]

                elevations.append(z)
                success = True
                break

            except Exception as e:
                wait = (2 ** attempt) + random.uniform(0, 1)
                print(f"node {idx} retry {attempt+1} wait {wait:.1f}s")
                time.sleep(wait)

        if not success:
            print(f"node {idx} failed permanently")
            elevations.append(None)

        # IMPORTANT : throttle global
        time.sleep(0.2 + random.random() * 0.3)

    nodes["elevation"] = elevations
    return nodes

In [ ]:
import osmnx as ox
import networkx as nx
import pandas as pd

from data.voirie import get_voirie
from data.filter import filter_impasses
from data.connectors import tag_connectors


# -------------------------
# GRAPH
# -------------------------
G = get_voirie("2nd arrondissement, Paris, France")
G_propre = filter_impasses(G)
G_tagged, nodes_gdf, edges_gdf = tag_connectors(
    G_propre,
    connector_length_m=12,
    min_degree=3,
)

# -------------------------
# ELEVATIONS
# -------------------------
nodes_gdf = fetch_ign_elevations_node_by_node(nodes_gdf)

elev_dict = {
    k: v for k, v in nodes_gdf["elevation"].to_dict().items()
    if pd.notna(v)
}

nx.set_node_attributes(G_tagged, elev_dict, "elevation")

# -------------------------
# GRADES
# -------------------------
G_tagged = ox.elevation.add_edge_grades(
    G_tagged,
    add_absolute=True
)

# -------------------------
# EXPORT
# -------------------------
nodes, edges = ox.graph_to_gdfs(G_tagged)

In [26]:
import networkx as nx
import numpy as np

for u, v, k, data in G_tagged.edges(keys=True, data=True):

    z_u = G_tagged.nodes[u].get("elevation")
    z_v = G_tagged.nodes[v].get("elevation")

    length = data.get("length", None)

    if z_u is None or z_v is None or length is None or length == 0:
        data["grade"] = None
        data["grade_abs"] = None
        continue

    grade = (z_v - z_u) / length

    data["grade"] = grade
    data["grade_abs"] = abs(grade)

In [27]:
nodes, edges = ox.graph_to_gdfs(G_tagged)

edges[["length", "grade", "grade_abs"]].describe()

,length,grade,grade_abs
count,806.000000,806.000000,806.000000
mean,53.300450,0.000483,0.010682
std,45.758101,0.015521,0.011265
min,1.177979,-0.085880,0.000000
25%,14.739577,-0.007126,0.003272
50%,41.133502,0.000651,0.007360
75%,79.815338,0.007432,0.013806
max,300.028076,0.079486,0.085880


In [28]:
import numpy as np

def road_traffic_penalty(highway):

    if isinstance(highway, list):
        highway = highway[0]

    mapping = {
        "motorway": 5,
        "trunk": 5,
        "primary": 4,
        "secondary": 3,
        "tertiary": 2,
        "residential": 1,
        "living_street": 0.5,
        "cycleway": 0.1,
        "path": 0.2,
        "service": 0.5,
    }

    return mapping.get(highway, 2)

In [29]:
for u, v, k, data in G_tagged.edges(keys=True, data=True):

    hw = data.get("highway")

    data["traffic_cost"] = road_traffic_penalty(hw)